# 5. Model Training

Uses labels taken from LLM and embeddings calculated earlier, and trains a ML model to do classification/regression. To be run on Kaggle or locally

Steps:
1. Add service_account.json secret with a service account with access to R2.
2. Disable accelerator
3. Save and commit so it runs on the background

Takes a bit over an hour

TODO next time: make sure the outputs are within 0, 1

In [ ]:
import io
import json
import re
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import polars as pl
import s3fs  # ty:ignore[unresolved-import]
import torch  # ty:ignore[unresolved-import]
import torch.nn as nn  # ty:ignore[unresolved-import]
from kaggle_secrets import UserSecretsClient  # ty:ignore[unresolved-import]
from sklearn.metrics import f1_score, precision_score, recall_score  # ty:ignore[unresolved-import]
from sklearn.model_selection import StratifiedShuffleSplit  # ty:ignore[unresolved-import]
from torch.utils.data import DataLoader, Dataset  # ty:ignore[unresolved-import]
from tqdm import tqdm

user_secrets = UserSecretsClient()
r2_creds = json.loads(user_secrets.get_secret("service_account.json"))
print("Authenticated with R2 access key:", r2_creds.get("access_key_id"))

R2_EMBEDDINGS = "keywordsdb/embeddings/raw"
R2_MODELS = "keywordsdb/models"
R2_LABELS = "keywordsdb/training/labels.parquet"
R2_TRAINING_CACHE = "keywordsdb/training/cache.parquet"

EMBEDDING_DIM = 384
TRAIN_TEST_SPLIT = 0.1

INTENT_LABELS = ["informational", "commercial", "transactional", "navigational"]
ENTITY_LABELS = ["person", "event", "location", "company", "product", "other"]

QUESTION_WORDS = re.compile(
    r"^(how|what|why|where|when|which|who|whom|whose|is|are|can|could|does|do|will|should)\b", re.IGNORECASE
)
VS_PATTERN = re.compile(r"\b(vs\.?|versus|compared to|comparison)\b", re.IGNORECASE)

FS = s3fs.S3FileSystem(
    key=r2_creds["access_key_id"], secret=r2_creds["secret_access_key"], endpoint_url=r2_creds["endpoint_url"]
)

In [ ]:
def _get_reduced_files() -> list[str]:
    return sorted(f.split("/")[-1] for f in FS.glob(f"{R2_EMBEDDINGS}/*.npz"))


def _npz_load(path: str) -> dict:
    with FS.open(path, "rb") as f:
        data = np.load(io.BytesIO(f.read()), allow_pickle=False)
        return dict(data)


def _load_labels() -> pl.DataFrame:
    with FS.open(R2_LABELS, "rb") as f:
        return pl.read_parquet(f.read())


def _save_torch(state_dict: dict, name: str) -> None:
    buf = io.BytesIO()
    torch.save(state_dict, buf)
    buf.seek(0)
    r2_path = f"{R2_MODELS}/mlp/{name}.pt"
    with FS.open(r2_path, "wb") as f:
        f.write(buf.read())
    print(f"Saved: {r2_path}")


def _split(
    x: np.ndarray, y: np.ndarray, stratify: bool = False
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    n = len(x)
    if stratify:
        sss = StratifiedShuffleSplit(n_splits=1, test_size=TRAIN_TEST_SPLIT, random_state=42)
        train_idx, test_idx = next(sss.split(x, y.astype(int)))
        return x[train_idx], x[test_idx], y[train_idx], y[test_idx]
    n_test = int(n * TRAIN_TEST_SPLIT)
    rng = np.random.RandomState(42)
    idx = rng.permutation(n)
    return x[idx[n_test:]], x[idx[:n_test]], y[idx[n_test:]], y[idx[:n_test]]


def _cls_metrics(y_true: np.ndarray, y_pred: np.ndarray, multiclass: bool = False) -> dict[str, float]:
    """Compute accuracy, macro/binary precision, recall, and F1 from integer predictions."""
    y_true_int = y_true.astype(int)
    y_pred_int = y_pred.astype(int)
    avg = "macro" if multiclass else "binary"
    return {
        "accuracy": float((y_pred_int == y_true_int).mean()),
        "precision": float(precision_score(y_true_int, y_pred_int, average=avg, zero_division=0)),
        "recall": float(recall_score(y_true_int, y_pred_int, average=avg, zero_division=0)),
        "f1": float(f1_score(y_true_int, y_pred_int, average=avg, zero_division=0)),
    }


def _optimal_threshold(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    """Find binary threshold that maximises F1 on the given data."""
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.1, 0.9, 0.01):
        preds = (y_prob > t).astype(int)
        f = float(f1_score(y_true.astype(int), preds, zero_division=0))
        if f > best_f1:
            best_f1 = f
            best_t = float(t)
    return best_t


def _text_features(keywords: list[str]) -> np.ndarray:
    """Compute simple text features from keyword strings."""
    feats = np.zeros((len(keywords), 5), dtype=np.float32)
    for i, kw in enumerate(keywords):
        feats[i, 0] = len(kw)  # char_count
        feats[i, 1] = len(kw.split())  # word_count
        feats[i, 2] = float(bool(QUESTION_WORDS.search(kw)))  # starts_with_question_word
        feats[i, 3] = float(bool(VS_PATTERN.search(kw)))  # contains_vs_or_comparison
        feats[i, 4] = float(bool(re.search(r"\d", kw)))  # has_digits
    return feats

In [ ]:
if FS.exists(R2_TRAINING_CACHE):
    print(f"Cache found — loading from {R2_TRAINING_CACHE}")
    with FS.open(R2_TRAINING_CACHE, "rb") as f:
        merged = pl.read_parquet(f.read())
    print(f"Loaded from cache: {len(merged)} rows, columns={merged.columns}")
else:
    print("No cache found — loading labels and embeddings from scratch")

    # Load labeled keywords first — this is small
    labeled = _load_labels()
    labeled = labeled.drop("has_purchase_intent")
    print(f"Loaded labels: rows={len(labeled)}, columns={labeled.columns}")

    labeled_keywords = set(labeled["keyword"].to_list())

    # Download and filter each embedding file in parallel (I/O-bound → threadpool)
    reduced_files = _get_reduced_files()
    print(f"Reduced embedding files: {len(reduced_files)}")

    def _load_and_filter(name: str) -> tuple[list[str], np.ndarray] | None:
        data = _npz_load(f"{R2_EMBEDDINGS}/{name}")
        keywords = data["keywords"].astype(str)
        mask = np.isin(keywords, list(labeled_keywords))
        if not mask.any():
            return None
        return keywords[mask].tolist(), data["embeddings"][mask].astype(np.float32)

    matched_keywords: list[str] = []
    matched_embeddings: list[np.ndarray] = []

    with ThreadPoolExecutor(max_workers=8) as pool:
        futures = {pool.submit(_load_and_filter, name): name for name in reduced_files}
        for future in tqdm(as_completed(futures), total=len(futures), desc="Loading embeddings"):
            result = future.result()
            if result is not None:
                kws, embs = result
                matched_keywords.extend(kws)
                matched_embeddings.append(embs)

    all_emb = pl.DataFrame(
        {"keyword": matched_keywords, "embedding": np.concatenate(matched_embeddings, axis=0).tolist()}
    )
    print(f"Matched embedding rows: {len(all_emb)}")

    merged = labeled.join(all_emb, on="keyword", how="inner")
    print(f"Merged rows (labeled + embeddings): {len(merged)}")

    buf = io.BytesIO()
    merged.write_parquet(buf)
    buf.seek(0)
    with FS.open(R2_TRAINING_CACHE, "wb") as f:
        f.write(buf.read())
    print(f"Saved training cache: {R2_TRAINING_CACHE}")

embeddings = np.array(merged["embedding"].to_list(), dtype=np.float32)
keywords_list = merged["keyword"].to_list()
text_feats = _text_features(keywords_list)
labels = merged.drop("embedding")
labels = labels.rename({"nsfw": "is_adult", "entity": "entity_type"})
print(f"Embedding matrix shape: {embeddings.shape}")
print(f"Text features shape: {text_feats.shape}")

In [ ]:
X = np.hstack([embeddings, text_feats])
print(f"Final feature matrix shape: {X.shape}  ({EMBEDDING_DIM} embedding dims + {text_feats.shape[1]} text features)")

## Multi-task MLP

Single shared backbone (128-dim PCA embeddings + 7 text features → 256 → 128) with separate
output heads per task. Trained jointly with masked losses so missing labels are ignored.


In [ ]:
# Build task config list: (task_key, col_name, task_type, n_outputs, label_names)
# task_key is used as the nn.ModuleDict key (no dots allowed)
TASK_CONFIGS = []

if "intent" in labels.columns:
    TASK_CONFIGS.append(("intent", "intent", "multiclass", len(INTENT_LABELS), INTENT_LABELS))
if "entity_type" in labels.columns:
    TASK_CONFIGS.append(("entity_type", "entity_type", "multiclass", len(ENTITY_LABELS), ENTITY_LABELS))
if "is_adult" in labels.columns:
    TASK_CONFIGS.append(("is_adult", "is_adult", "regression", 1, None))
if "saas_potential" in labels.columns:
    TASK_CONFIGS.append(("saas_potential", "saas_potential", "regression", 1, None))
for flag in ["is_question", "is_comparison", "is_branded"]:
    col = f"flags.{flag}" if f"flags.{flag}" in labels.columns else flag
    if col in labels.columns:
        TASK_CONFIGS.append((flag, col, "binary", 1, None))

print(f"MLP tasks: {[t[0] for t in TASK_CONFIGS]}")


class MultiTaskMLP(nn.Module):
    def __init__(self, input_dim: int, task_configs: list) -> None:
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.LayerNorm(1024),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.1),
        )
        self.heads = nn.ModuleDict({task_key: nn.Linear(256, n_out) for task_key, _, _, n_out, _ in task_configs})

    def forward(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        h = self.backbone(x)
        return {task_key: self.heads[task_key](h) for task_key in self.heads}


class KeywordDataset(Dataset):
    def __init__(self, x: np.ndarray, task_labels: dict[str, np.ndarray]) -> None:
        self.x = torch.from_numpy(x).float()
        self.labels = {k: torch.from_numpy(v).float() for k, v in task_labels.items()}

    def __len__(self) -> int:
        return len(self.x)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
        return self.x[idx], {k: v[idx] for k, v in self.labels.items()}

In [ ]:
MLP_EPOCHS = 150
MLP_BATCH_SIZE = 1024
MLP_LR = 1e-3

# Build a unified label array per task (float32, NaN or -1 for missing)
task_y: dict[str, np.ndarray] = {}
for task_key, col_name, task_type, _, label_names in TASK_CONFIGS:
    if task_type == "multiclass":
        assert isinstance(label_names, list)
        lmap = {n: i for i, n in enumerate(label_names)}
        arr = (
            labels[col_name]
            .map_elements(lambda v: lmap.get(v, -1), return_dtype=pl.Int32)
            .to_numpy()
            .astype(np.float32)
        )
    else:
        arr = labels[col_name].cast(pl.Float32).to_numpy()
    task_y[task_key] = arr

# Train / test split (same seed as other methods)
n = len(X)
n_test = int(n * TRAIN_TEST_SPLIT)
rng = np.random.RandomState(42)
idx = rng.permutation(n)
train_idx, test_idx = idx[n_test:], idx[:n_test]

# Precompute per-task pos_weight for binary BCE (on training set)
binary_pos_weights: dict[str, torch.Tensor] = {}
for task_key, _, task_type, _, _ in TASK_CONFIGS:
    if task_type == "binary":
        y_train = task_y[task_key][train_idx]
        valid = y_train[~np.isnan(y_train)]
        n_pos = (valid == 1).sum()
        n_neg = (valid == 0).sum()
        pw = float(n_neg) / float(n_pos) if n_pos > 0 else 1.0
        binary_pos_weights[task_key] = torch.tensor([pw], device="cuda")
        print(f"  {task_key}: pos_weight={pw:.2f}  (pos={n_pos}, neg={n_neg})")

# Precompute per-task class weights for multiclass CE (on training set)
multiclass_weights: dict[str, torch.Tensor] = {}
for task_key, _, task_type, _, task_label_names in TASK_CONFIGS:
    if task_type == "multiclass":
        assert task_label_names
        y_train = task_y[task_key][train_idx]
        valid = y_train[y_train >= 0].astype(int)
        classes, counts = np.unique(valid, return_counts=True)
        w = np.ones(len(task_label_names), dtype=np.float32)
        for c, cnt in zip(classes, counts):
            w[c] = len(valid) / (len(classes) * cnt)
        multiclass_weights[task_key] = torch.from_numpy(w).to("cuda")
        print(f"  {task_key}: class_weights={dict(zip(task_label_names, w.round(3)))}")

MLP_MODEL_PATH = f"{R2_MODELS}/mlp/model.pt"
mlp = MultiTaskMLP(X.shape[1], TASK_CONFIGS).to("cuda")

if FS.exists(MLP_MODEL_PATH):
    print(f"MLP model found in cache — loading from {MLP_MODEL_PATH}")
    with FS.open(MLP_MODEL_PATH, "rb") as f:
        state_dict = torch.load(io.BytesIO(f.read()), map_location="cuda")
    mlp.load_state_dict(state_dict)
else:
    train_ds = KeywordDataset(X[train_idx], {k: v[train_idx] for k, v in task_y.items()})
    train_loader = DataLoader(train_ds, batch_size=MLP_BATCH_SIZE, shuffle=True, pin_memory=True)

    optimizer = torch.optim.AdamW(mlp.parameters(), lr=MLP_LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MLP_EPOCHS)

    # Per-task loss functions with class weighting
    ce_fns = {}
    for task_key in multiclass_weights:
        ce_fns[task_key] = nn.CrossEntropyLoss(weight=multiclass_weights[task_key], label_smoothing=0.1)
    mse_fn = nn.MSELoss()
    bce_fns = {}
    for task_key in binary_pos_weights:
        bce_fns[task_key] = nn.BCEWithLogitsLoss(pos_weight=binary_pos_weights[task_key])

    print(
        f"Training MLP: {sum(p.numel() for p in mlp.parameters()):,} params, "
        f"{len(train_ds):,} train samples, {MLP_EPOCHS} epochs"
    )

    for epoch in range(MLP_EPOCHS):
        mlp.train()
        epoch_loss = 0.0
        for x_b, y_b in train_loader:
            x_b = x_b.to("cuda")
            optimizer.zero_grad()
            preds = mlp(x_b)
            loss = torch.zeros(1, device="cuda")
            for task_key, _, task_type, _, _ in TASK_CONFIGS:
                if task_key not in y_b:
                    continue
                y = y_b[task_key].to("cuda")
                p = preds[task_key]
                if task_type == "multiclass":
                    mask = y.long() >= 0
                    if mask.sum() == 0:
                        continue
                    loss = loss + ce_fns[task_key](p[mask], y[mask].long())
                elif task_type == "regression":
                    mask = ~y.isnan()
                    if mask.sum() == 0:
                        continue
                    loss = loss + mse_fn(p[mask].squeeze(-1), y[mask])
                else:  # binary
                    mask = ~y.isnan()
                    if mask.sum() == 0:
                        continue
                    loss = loss + bce_fns[task_key](p[mask].squeeze(-1), y[mask])
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        scheduler.step()
        if (epoch + 1) % 15 == 0:
            print(f"  Epoch {epoch + 1}/{MLP_EPOCHS}  loss={epoch_loss / len(train_loader):.4f}")

    _save_torch(mlp.state_dict(), "model")

# Evaluate on held-out test set (always runs, whether loaded or trained)
mlp.eval()
mlp_metrics: dict[str, dict[str, float]] = {}
with torch.no_grad():
    x_test_t = torch.from_numpy(X[test_idx]).float().to("cuda")
    test_preds = mlp(x_test_t)

for task_key, _, task_type, _, _ in TASK_CONFIGS:
    y_test = task_y[task_key][test_idx]
    p = test_preds[task_key].cpu().numpy()
    if task_type == "multiclass":
        mask = y_test >= 0
        y_pred = p[mask].argmax(axis=1)
        metrics = _cls_metrics(y_test[mask], y_pred, multiclass=True)
        mlp_metrics[task_key] = metrics
        print(
            f"[mlp {task_key}] accuracy={metrics['accuracy']:.4f}  precision={metrics['precision']:.4f}"
            f"  recall={metrics['recall']:.4f}  f1={metrics['f1']:.4f}"
        )
    elif task_type == "regression":
        mask = ~np.isnan(y_test)
        rmse = float(np.sqrt(((p[mask].squeeze() - y_test[mask]) ** 2).mean()))
        mlp_metrics[task_key] = {"rmse": rmse}
        print(f"[mlp {task_key}] RMSE={rmse:.4f}")
    else:  # binary
        mask = ~np.isnan(y_test)
        y_prob = 1.0 / (1.0 + np.exp(-p[mask].squeeze()))  # sigmoid
        threshold = _optimal_threshold(y_test[mask], y_prob)
        y_pred = (y_prob > threshold).astype(int)
        metrics = _cls_metrics(y_test[mask], y_pred)
        metrics["threshold"] = threshold
        mlp_metrics[task_key] = metrics
        print(
            f"[mlp {task_key}] accuracy={metrics['accuracy']:.4f}  precision={metrics['precision']:.4f}"
            f"  recall={metrics['recall']:.4f}  f1={metrics['f1']:.4f}  threshold={threshold:.2f}"
        )

all_results: dict[str, dict[str, dict[str, float]]] = {}
all_results["mlp"] = mlp_metrics
print(f"\nMLP done: {list(mlp_metrics.keys())}")

## Results Comparison


In [ ]:
REGRESSION_TASKS = {"is_adult", "saas_potential"}
METHODS = ["mlp"]

all_tasks = sorted({task for results in all_results.values() for task in results})


def _task_metric_keys(task: str) -> list[str]:
    return ["rmse"] if task in REGRESSION_TASKS else ["accuracy", "precision", "recall", "f1"]


# Build comparison table: one row per (task, metric)
rows = []
for task in all_tasks:
    for metric in _task_metric_keys(task):
        row = {"task": task, "metric": metric}
        for method in METHODS:
            val = all_results.get(method, {}).get(task, {}).get(metric)
            row[method] = f"{val:.4f}" if val is not None else "—"
        rows.append(row)

df = pl.DataFrame(rows)
print("\nResults comparison:")
for task in all_tasks:
    for metric in _task_metric_keys(task):
        subset = df.filter((pl.col("task") == task) & (pl.col("metric") == metric)).drop(["task", "metric"])
        print(f"  {task:30s}  {metric:10s}  →  {subset.to_dicts()[0]}")

# Winner per (task, metric)
print("\nWinner per task / metric:")
for task in all_tasks:
    for metric in _task_metric_keys(task):
        scores = {m: all_results.get(m, {}).get(task, {}).get(metric) for m in METHODS}
        scores = {m: v for m, v in scores.items() if v is not None}
        if not scores:
            continue
        lower_is_better = metric == "rmse"
        winner = min(scores, key=scores.__getitem__) if lower_is_better else max(scores, key=scores.__getitem__)
        arrow = "↓" if lower_is_better else "↑"
        print(f"  {task:30s}  {metric:10s}  →  {winner}  ({scores[winner]:.4f}{arrow})")

# Print optimal thresholds for binary tasks
print("\nOptimal thresholds for binary tasks:")
for task in all_tasks:
    if task in REGRESSION_TASKS or task in {"intent", "entity_type"}:
        continue
    for method in METHODS:
        t = all_results.get(method, {}).get(task, {}).get("threshold")
        if t is not None:
            print(f"  {task:30s}  {method:10s}  →  {t:.2f}")